# Augment and Dedupe Receipts (one-time migration)

Backfills the missing `event_ts` / `event_date` columns on legacy Silver
`fact_receipts`, `fact_receipt_lines`, `fact_payments`, and
`fact_online_order_headers` rows and removes duplicate `receipt_id_ext` values.

## Why
1. Receipt headers loaded before the v2 silver transform have no `event_ts`,
   so the semantic model has no relationship to `dim_date` and every
   time-intelligence measure (YTD, PY, YoY, MoM, QoQ, DoD) collapses to the
   same value.
2. Legacy generator runs produced duplicate `receipt_id_ext` values
   (e.g. `RCP2024120113180125587`) which break the
   `fact_receipts -> fact_receipt_lines / fact_payments` one-to-many
   relationships in Direct Lake.

## How
The legacy receipt identifier format encodes the event time:
```
  RCP YYYYMMDDHHMM <store_id padded> <sequence>
  e.g. RCP 202412011318 01 25587
```
We extract the timestamp from positions 4-15, build `event_ts` /
`event_date`, dedupe receipts by `receipt_id_ext` (keeping the earliest
ingested row), then propagate the same `event_ts` / `event_date` to
`fact_receipt_lines` and `fact_payments` via a join on `receipt_id_ext`.

## Usage
Run **once** in Fabric against the `retail_lakehouse` Lakehouse, then
refresh the `retail_model` semantic model. Safe to re-run (idempotent).

## Column Naming Convention
All column names use `snake_case` (see CLAUDE.md).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
import os

In [ ]:
SILVER_DB = os.environ.get("SILVER_DB", "silver")

RECEIPTS_TABLE = f"{SILVER_DB}.fact_receipts"
LINES_TABLE = f"{SILVER_DB}.fact_receipt_lines"
PAYMENTS_TABLE = f"{SILVER_DB}.fact_payments"
ONLINE_ORDERS_TABLE = f"{SILVER_DB}.fact_online_order_headers"

print(f"Targeting tables in {SILVER_DB}: fact_receipts, fact_receipt_lines, fact_payments, fact_online_order_headers")

In [ ]:
def parse_event_ts_from_receipt_id(col):
    """Receipt id format: RCP{YYYYMMDDHHMM}{store}{seq}.

    Returns a timestamp parsed from chars 4..15 (1-indexed for Spark substring).
    Falls back to NULL when the prefix does not match.
    """
    ts_string = F.when(
        F.col(col).rlike("^RCP[0-9]{12}"),
        F.substring(F.col(col), 4, 12)
    )
    return F.to_timestamp(ts_string, "yyyyMMddHHmm")

## Step 1 — Augment + dedupe `fact_receipts`

In [ ]:
receipts = spark.read.table(RECEIPTS_TABLE)
print(f"Loaded {RECEIPTS_TABLE} ({receipts.count():,} rows)")
print("Columns before:", receipts.columns)

if "event_ts" not in receipts.columns:
    receipts = receipts.withColumn("event_ts", parse_event_ts_from_receipt_id("receipt_id_ext"))
else:
    receipts = receipts.withColumn(
        "event_ts",
        F.coalesce(F.col("event_ts"), parse_event_ts_from_receipt_id("receipt_id_ext")),
    )

if "event_date" not in receipts.columns:
    receipts = receipts.withColumn("event_date", F.to_date(F.col("event_ts")))
else:
    receipts = receipts.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

null_dates = receipts.filter(F.col("event_date").isNull()).count()
print(f"Rows with null event_date after augmentation: {null_dates:,}")

In [ ]:
from pyspark.sql.window import Window

dup_pre = (
    receipts.groupBy("receipt_id_ext").count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate receipt_id_ext values before dedupe: {dup_pre:,}")

window = Window.partitionBy("receipt_id_ext").orderBy(F.col("event_ts").asc_nulls_last())
deduped = (
    receipts
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

if "__index_level_0__" in deduped.columns:
    deduped = deduped.drop("__index_level_0__")

print(f"Rows after dedupe: {deduped.count():,}")

(
    deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RECEIPTS_TABLE)
)
print(f"Rewrote {RECEIPTS_TABLE} with event_ts + event_date + dedupe applied.")

## Step 2 — Propagate `event_ts` / `event_date` to `fact_receipt_lines`

In [ ]:
lines = spark.read.table(LINES_TABLE)
print(f"Loaded {LINES_TABLE} ({lines.count():,} rows)")
print("Columns before:", lines.columns)

header_dates = (
    spark.read.table(RECEIPTS_TABLE)
    .select("receipt_id_ext", F.col("event_ts").alias("_hdr_event_ts"), F.col("event_date").alias("_hdr_event_date"))
)

lines_joined = lines.join(header_dates, on="receipt_id_ext", how="left")

if "event_ts" not in lines.columns:
    lines_aug = lines_joined.withColumn("event_ts", F.col("_hdr_event_ts"))
else:
    lines_aug = lines_joined.withColumn("event_ts", F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")))

if "event_date" not in lines.columns:
    lines_aug = lines_aug.withColumn("event_date", F.col("_hdr_event_date"))
else:
    lines_aug = lines_aug.withColumn("event_date", F.coalesce(F.col("event_date"), F.col("_hdr_event_date")))

lines_aug = lines_aug.drop("_hdr_event_ts", "_hdr_event_date")
if "__index_level_0__" in lines_aug.columns:
    lines_aug = lines_aug.drop("__index_level_0__")

missing = lines_aug.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after join: {missing:,}")

(
    lines_aug.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(LINES_TABLE)
)
print(f"Rewrote {LINES_TABLE} with event_ts + event_date columns.")

## Step 3 — Propagate `event_ts` / `event_date` to `fact_payments`

In [ ]:
payments = spark.read.table(PAYMENTS_TABLE)
print(f"Loaded {PAYMENTS_TABLE} ({payments.count():,} rows)")
print("Columns before:", payments.columns)

header_dates2 = (
    spark.read.table(RECEIPTS_TABLE)
    .select("receipt_id_ext", F.col("event_ts").alias("_hdr_event_ts"), F.col("event_date").alias("_hdr_event_date"))
)

payments_joined = payments.join(header_dates2, on="receipt_id_ext", how="left")

if "event_ts" not in payments.columns:
    payments_aug = payments_joined.withColumn("event_ts", F.col("_hdr_event_ts"))
else:
    payments_aug = payments_joined.withColumn("event_ts", F.coalesce(F.col("event_ts"), F.col("_hdr_event_ts")))

if "event_date" not in payments.columns:
    payments_aug = payments_aug.withColumn("event_date", F.col("_hdr_event_date"))
else:
    payments_aug = payments_aug.withColumn("event_date", F.coalesce(F.col("event_date"), F.col("_hdr_event_date")))

payments_aug = payments_aug.drop("_hdr_event_ts", "_hdr_event_date")
if "__index_level_0__" in payments_aug.columns:
    payments_aug = payments_aug.drop("__index_level_0__")

missing2 = payments_aug.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after join: {missing2:,}")

(
    payments_aug.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PAYMENTS_TABLE)
)
print(f"Rewrote {PAYMENTS_TABLE} with event_ts + event_date columns.")

## Step 3b — Backfill `event_date` on `fact_online_order_headers`

Online order headers always carry `event_ts`, but legacy rows predate the
`event_date` column the semantic model now uses for its `dim_date`
relationship (Direct Lake requires date-to-date relationship columns).


In [ ]:
orders = spark.read.table(ONLINE_ORDERS_TABLE)
print(f"Loaded {ONLINE_ORDERS_TABLE} ({orders.count():,} rows)")
print("Columns before:", orders.columns)

if "event_date" not in orders.columns:
    orders_aug = orders.withColumn("event_date", F.to_date(F.col("event_ts")))
else:
    orders_aug = orders.withColumn(
        "event_date",
        F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))),
    )

if "__index_level_0__" in orders_aug.columns:
    orders_aug = orders_aug.drop("__index_level_0__")

missing_orders = orders_aug.filter(F.col("event_date").isNull()).count()
print(f"Rows missing event_date after backfill: {missing_orders:,}")

(
    orders_aug.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ONLINE_ORDERS_TABLE)
)
print(f"Rewrote {ONLINE_ORDERS_TABLE} with event_date backfilled.")


## Step 3c — Normalize legacy PascalCase columns on promotions / foot traffic / store ops

Early datagen exports wrote PascalCase column names (`PromoCode`, `StoreID`,
`Count`, ...) while the v2 pipeline writes snake_case. Tables loaded from the
legacy exports therefore don't match the semantic model (snake_case per the
project convention), which breaks the Promotions / Foot Traffic / Store
Operations visuals. This step renames legacy columns (coalescing when both
spellings exist), backfills `event_ts` / `event_date`, and drops the pandas
index column.


In [ ]:
LEGACY_RENAMES = {
    f"{SILVER_DB}.fact_promotions": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "ReceiptId": "receipt_id_ext",
        "PromoCode": "promo_code",
        "DiscountAmount": "discount_amount",
        "DiscountCents": "discount_cents",
        "DiscountType": "discount_type",
        "ProductCount": "product_count",
        "ProductIds": "product_ids",
        "StoreID": "store_id",
        "CustomerID": "customer_id",
    },
    f"{SILVER_DB}.fact_foot_traffic": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "SensorId": "sensor_id",
        "Zone": "zone",
        "Dwell": "dwell_seconds",
        "Count": "count",
    },
    f"{SILVER_DB}.fact_store_ops": {
        "EventTS": "event_ts",
        "TraceId": "trace_id",
        "StoreID": "store_id",
        "OperationTime": "operation_time",
        "OperationType": "operation_type",
    },
}

for table, renames in LEGACY_RENAMES.items():
    try:
        df = spark.read.table(table)
    except AnalysisException:
        print(f"Skipping {table}: table does not exist")
        continue

    print(f"\n=== {table} ===")
    print("Columns before:", df.columns)

    for legacy, snake in renames.items():
        if legacy not in df.columns:
            continue
        if snake in df.columns:
            # both spellings exist (legacy load + v2 streaming): coalesce into snake_case
            df = df.withColumn(snake, F.coalesce(F.col(snake), F.col(legacy))).drop(legacy)
        else:
            df = df.withColumnRenamed(legacy, snake)

    # store_ops legacy rows have operation_time but no event_ts
    if "event_ts" in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.coalesce(F.col("event_ts"), F.col("operation_time")))
    elif "event_ts" not in df.columns and "operation_time" in df.columns:
        df = df.withColumn("event_ts", F.col("operation_time"))

    if "event_date" in df.columns:
        df = df.withColumn("event_date", F.coalesce(F.col("event_date"), F.to_date(F.col("event_ts"))))
    elif "event_ts" in df.columns:
        df = df.withColumn("event_date", F.to_date(F.col("event_ts")))

    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    print("Columns after:", spark.read.table(table).columns)
    print(f"Rewrote {table}.")


## Step 4 — Verification

In [ ]:
for table in (RECEIPTS_TABLE, LINES_TABLE, PAYMENTS_TABLE, ONLINE_ORDERS_TABLE):
    df = spark.read.table(table)
    print(f"\n=== {table} ===")
    print("Columns:", df.columns)
    print("Rows:", df.count())
    if "event_date" in df.columns:
        df.agg(
            F.min("event_date").alias("min_date"),
            F.max("event_date").alias("max_date"),
            F.sum(F.when(F.col("event_date").isNull(), 1).otherwise(0)).alias("null_dates"),
        ).show()

receipts_final = spark.read.table(RECEIPTS_TABLE)
remaining_dupes = (
    receipts_final.groupBy("receipt_id_ext").count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"\nDuplicate receipt_id_ext remaining: {remaining_dupes:,}")
assert remaining_dupes == 0, "Duplicate receipt_id_ext values remain."
print("\nReady. Refresh the retail_model semantic model in Fabric to pick up the new columns.")